# Notebook 06 — Multi-Seed LoRA r=16 Replication

**Objective:** Re-train LoRA r=16 with three random seeds and report mean ± std for macro F1 and AUC-ROC, so the dissertation can attach seed-variance error bars to its headline result (report §5, limitation 1).

**Runtime:** ~50 min on a T4/P100 (3 × ~16 min). GPU required: Runtime → Change runtime type → GPU.

**Output:** `dissertation_results/lora_r16_multiseed.csv` on Drive + a summary line to paste into the report.


In [ ]:
# Section 1 — Mount Drive & install
# NOTE: if you already ran the old install cell and imports failed with a
# torch._dynamo AttributeError, do Runtime -> Restart runtime, then run this cell first.
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Same dependency recipe that works in Notebook 05:
# clean out the torchao that clashes with Colab's preinstalled torch,
# then install a compatible version alongside peft + timm.
!pip uninstall -y -q torchao
!pip install -q "torchao>=0.16.0" peft timm
print('Ready. If imports in Section 2 still fail: Runtime -> Restart runtime, run Sections 1-2 again.')


In [ ]:
# Section 2 — Imports & config
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import timm, time, warnings
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from peft import LoraConfig, get_peft_model
from sklearn.metrics import f1_score, roc_auc_score
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type == 'cuda', 'Enable a GPU runtime first!'
print('GPU:', torch.cuda.get_device_name(0))

DATASET_PATH = Path('/content/drive/MyDrive/dessertation/kepler_gaf_dataset.npz')
RESULTS_DIR  = Path('/content/drive/MyDrive/dessertation/dissertation_results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEEDS  = [42, 123, 2026]
RANK   = 16
EPOCHS = 10
LR     = 2e-4
BATCH  = 32


In [ ]:
# Section 3 — Data (identical to Notebook 02)
data = np.load(DATASET_PATH)
X_train, y_train = data['X_train'], data['y_train']
X_val,   y_val   = data['X_val'],   data['y_val']
X_test,  y_test  = data['X_test'],  data['y_test']

n_conf = y_train.sum(); n_fp = len(y_train) - n_conf
CLASS_WEIGHTS = torch.tensor([len(y_train)/(2.0*n_fp), len(y_train)/(2.0*n_conf)],
                             dtype=torch.float32).to(DEVICE)

class GAFDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).unsqueeze(1); self.y = torch.from_numpy(y).long()
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        img = F.interpolate(self.X[idx].unsqueeze(0), size=224, mode='bilinear',
                            align_corners=False).squeeze(0).repeat(3,1,1)
        return (img - 0.5) / 0.5, self.y[idx]

train_ds, val_ds, test_ds = GAFDataset(X_train,y_train), GAFDataset(X_val,y_val), GAFDataset(X_test,y_test)
val_loader  = DataLoader(val_ds,  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
print('Data loaded:', X_train.shape, X_val.shape, X_test.shape)


In [ ]:
# Section 4 — Train/evaluate helpers (identical hyperparameters to Notebook 02)
def evaluate(model, loader):
    model.eval(); P, Pr, L = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            logits = model(imgs.to(DEVICE))
            Pr.append(torch.softmax(logits,1)[:,1].cpu()); P.append(logits.argmax(1).cpu()); L.append(labels)
    P, Pr, L = torch.cat(P).numpy(), torch.cat(Pr).numpy(), torch.cat(L).numpy()
    return f1_score(L, P, average='macro'), roc_auc_score(L, Pr)

def train_one_seed(seed):
    torch.manual_seed(seed); np.random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2,
                              pin_memory=True, generator=torch.Generator().manual_seed(seed))
    base = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=2)
    cfg = LoraConfig(r=RANK, lora_alpha=RANK*2, target_modules=['qkv'],
                     lora_dropout=0.1, bias='none', modules_to_save=['head'])
    model = get_peft_model(base, cfg).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    crit = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)
    best_f1, best_state = 0.0, None
    for ep in range(1, EPOCHS+1):
        model.train()
        for imgs, labels in train_loader:
            opt.zero_grad()
            loss = crit(model(imgs.to(DEVICE)), labels.to(DEVICE))
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        sched.step()
        vf1, vauc = evaluate(model, val_loader)
        print(f'  seed {seed} epoch {ep:02d}  val_F1={vf1:.4f}  val_AUC={vauc:.4f}')
        if vf1 > best_f1: best_f1, best_state = vf1, {k: v.clone() for k,v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model


In [ ]:
# Section 5 — Run all seeds
rows = []
for seed in SEEDS:
    print(f'===== SEED {seed} =====')
    t0 = time.time()
    model = train_one_seed(seed)
    f1, auc = evaluate(model, test_loader)
    mins = (time.time()-t0)/60
    print(f'  TEST  F1={f1:.4f}  AUC={auc:.4f}  ({mins:.1f} min)')
    rows.append({'seed': seed, 'f1_macro': round(f1,4), 'auc_roc': round(auc,4), 'train_min': round(mins,1)})
    del model; torch.cuda.empty_cache()

df = pd.DataFrame(rows)
df.to_csv(RESULTS_DIR/'lora_r16_multiseed.csv', index=False)
print(df)
print()
print('=== PASTE INTO REPORT ===')
print(f"LoRA r=16, 3 seeds: macro F1 = {df.f1_macro.mean():.3f} ± {df.f1_macro.std():.3f}, "
      f"AUC-ROC = {df.auc_roc.mean():.3f} ± {df.auc_roc.std():.3f}")
